In [1]:
cd ..

/home/dataflix/Projects/maji-ndogo-yield-intelligence


In [2]:
from src.config import config_params
from src.field_data_processor import FieldDataProcessor
field_df = FieldDataProcessor(config_params).process()
field_df.head()

2026-08-16 20:55:24 | src.data_ingestion | INFO | Starting data ingestion
2026-08-16 20:55:24 | src.field_data_processor | INFO | FieldDataProcessor is initialized
2026-08-16 20:55:24 | src.data_ingestion | INFO | Successfully connected to sqlite:///data/Maji_Ndogo_farm_survey_small.db
2026-08-16 20:55:24 | src.data_ingestion | INFO | Query executed successfully. Rows: 5654
2026-08-16 20:55:24 | src.field_data_processor | INFO | SQL data is successfully loaded into the pandas DataFrame
2026-08-16 20:55:24 | src.field_data_processor | INFO | Swapped columns: Annual_yield with Crop_type
2026-08-16 20:55:24 | src.field_data_processor | INFO | Converted the negative elavtion values to absulte figures.
2026-08-16 20:55:24 | src.field_data_processor | INFO | Mispelled crop names and extra spaces were found and got fixed
2026-08-16 20:55:24 | src.data_ingestion | INFO | Attempting to read CSV from: https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Maji_Ndogo/Weather_data_field_m

,Elevation,Latitude,Longitude,Location,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,Soil_type,pH,Pollution_level,Plot_size,Annual_yield,Crop_type,Standard_yield
0,786.05580,-7.389911,-7.556202,Rural_Akatsi,14.795113,1125.2,-3.1,33.1,15.00,0.62,Sandy,6.169393,0.085267,1.3,0.751354,cassava,0.577964
1,674.33410,-7.736849,-1.051539,Rural_Sokoto,11.374611,1450.7,-3.9,30.6,13.35,0.64,Volcanic,5.676648,0.399684,2.2,1.069865,cassava,0.486302
2,826.53390,-9.926616,0.115156,Rural_Sokoto,11.339692,2208.9,-1.8,28.4,13.30,0.69,Volcanic,5.331993,0.358029,3.4,2.208801,tea,0.649647
3,574.94617,-2.420131,-6.592215,Rural_Kilimani,7.109855,328.8,-5.8,32.2,13.20,0.54,Loamy,5.328150,0.286687,2.4,1.277635,cassava,0.532348
4,886.35300,-3.055434,-7.952609,Rural_Kilimani,55.007656,785.2,-2.5,31.0,14.25,0.72,Sandy,5.721234,0.043190,1.5,0.832614,wheat,0.555076


In [3]:
field_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5654 entries, 0 to 5653
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Elevation          5654 non-null   float64
 1   Latitude           5654 non-null   float64
 2   Longitude          5654 non-null   float64
 3   Location           5654 non-null   str    
 4   Slope              5654 non-null   float64
 5   Rainfall           5654 non-null   float64
 6   Min_temperature_C  5654 non-null   float64
 7   Max_temperature_C  5654 non-null   float64
 8   Ave_temps          5654 non-null   float64
 9   Soil_fertility     5654 non-null   float64
 10  Soil_type          5654 non-null   str    
 11  pH                 5654 non-null   float64
 12  Pollution_level    5654 non-null   float64
 13  Plot_size          5654 non-null   float64
 14  Annual_yield       5654 non-null   float64
 15  Crop_type          5654 non-null   str    
 16  Standard_yield     5654 non-null   

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
y = field_df['Standard_yield']
X = field_df.drop(columns =["Standard_yield", "Annual_yield"])


Annual_yield was excluded from the predictor set because it represents total field output and is a derived yield-related outcome rather than an independent environmental or agricultural predictor. Including it could introduce target leakage and would not reflect the intended prediction setting.

In [6]:
print(f"X shape:{X.shape}")
print(f"y shape {y.shape}")


X shape:(5654, 15)
y shape (5654,)


In [7]:
X.dtypes

Elevation            float64
Latitude             float64
Longitude            float64
Location                 str
Slope                float64
Rainfall             float64
Min_temperature_C    float64
Max_temperature_C    float64
Ave_temps            float64
Soil_fertility       float64
Soil_type                str
pH                   float64
Pollution_level      float64
Plot_size            float64
Crop_type                str
dtype: object

In [8]:
def training_testing_splits(X, y) -> np.ndarray:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = 0.2, random_state =42
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = training_testing_splits(X, y)


In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4523, 15)
X_test: (1131, 15)
y_train: (4523,)
y_test: (1131,)


In [10]:
def feature_types(X_train) -> np.ndarray:
    numerical_features = X_train.select_dtypes("float", "int").columns.tolist()
    categorical_features = X_train.select_dtypes("str").columns.tolist()
    return numerical_features, categorical_features
numerical_features, categorical_features = feature_types(X_train)
print("NUMERICAL FEATURES")
display(numerical_features)
print("CATEGORICAL FEATURES")
display(categorical_features)

NUMERICAL FEATURES


['Elevation',
 'Latitude',
 'Longitude',
 'Slope',
 'Rainfall',
 'Min_temperature_C',
 'Max_temperature_C',
 'Ave_temps',
 'Soil_fertility',
 'pH',
 'Pollution_level',
 'Plot_size']

CATEGORICAL FEATURES


['Location', 'Soil_type', 'Crop_type']

In [11]:
def X_encoder() ->np.ndarray:
    encoder = OneHotEncoder(
        drop = [
            ["Rural_Kilimani"],
            ["Sandy"],
            ["wheat"]
        ], handle_unknown="ignore")
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", encoder, categorical_features),
            ("numerical", "passthrough", numerical_features)
        ]
    )
    return encoder, preprocessor
encoder, preprocessor = X_encoder()



In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [14]:
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (4523, 28)
Processed X_test shape: (1131, 28)


In [22]:
feature_names = [
    name.replace("categorical__", "")
        .replace("numerical__", "")
    for name in preprocessor.get_feature_names_out()]
display(feature_names)
print("Number of features:", len(feature_names))

['Location_Rural_Akatsi',
 'Location_Rural_Amanzi',
 'Location_Rural_Hawassa',
 'Location_Rural_Sokoto',
 'Soil_type_Loamy',
 'Soil_type_Peaty',
 'Soil_type_Rocky',
 'Soil_type_Silt',
 'Soil_type_Volcanic',
 'Crop_type_banana',
 'Crop_type_cassava',
 'Crop_type_coffee',
 'Crop_type_maize',
 'Crop_type_potato',
 'Crop_type_rice',
 'Crop_type_tea',
 'Elevation',
 'Latitude',
 'Longitude',
 'Slope',
 'Rainfall',
 'Min_temperature_C',
 'Max_temperature_C',
 'Ave_temps',
 'Soil_fertility',
 'pH',
 'Pollution_level',
 'Plot_size']

Number of features: 28


In [26]:
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index = X_train.index)
X_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index = X_test.index)

In [29]:
X_train_df.head()

,Location_Rural_Akatsi,Location_Rural_Amanzi,Location_Rural_Hawassa,Location_Rural_Sokoto,Soil_type_Loamy,Soil_type_Peaty,Soil_type_Rocky,Soil_type_Silt,Soil_type_Volcanic,Crop_type_banana,...,Longitude,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,pH,Pollution_level,Plot_size
670,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.970481,3.717101,1186.7,-4.4,30.8,13.20,0.59,5.659775,5.764306e-01,14.1
3484,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,-6.700326,3.211858,1573.0,-5.9,30.7,12.40,0.62,6.361063,3.918987e-08,12.1
2767,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.488462,9.095326,1553.8,-3.0,33.6,15.30,0.64,5.275695,4.634201e-01,4.9
1620,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-3.889446,7.851821,1175.3,-4.3,30.5,13.10,0.61,6.111260,3.585489e-02,4.1
3606,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,-4.186751,5.384154,735.6,-2.6,33.1,15.25,0.56,5.040559,2.092824e-01,2.8


In [30]:
X_test_df.head()

,Location_Rural_Akatsi,Location_Rural_Amanzi,Location_Rural_Hawassa,Location_Rural_Sokoto,Soil_type_Loamy,Soil_type_Peaty,Soil_type_Rocky,Soil_type_Silt,Soil_type_Volcanic,Crop_type_banana,...,Longitude,Slope,Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,pH,Pollution_level,Plot_size
4816,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,-7.015649,8.106139,1562.8,-8.6,36.3,13.85,0.64,5.572066,0.001636,4.4
5096,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-3.988548,7.534660,466.3,-6.4,32.3,12.95,0.55,5.098073,0.019206,4.0
4706,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.992979,23.671692,1467.5,-2.7,28.6,12.95,0.68,5.423997,0.057128,1.8
1499,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-2.560585,11.226282,983.7,-3.0,28.9,12.95,0.60,6.691951,0.391307,4.0
3544,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-7.623772,14.173515,555.1,-4.5,31.6,13.55,0.58,4.938555,0.333970,1.8


In [35]:
print(f"X_train_df shape: {X_train_df.shape}")
print(f"X_test_df shape: {X_test_df.shape}")

X_train_df shape: (4523, 28)
X_test_df shape: (1131, 28)


In [ ]:
def add_constant(X_train_df, X_test_df)->pd.DataFrame:
    X_train_ols = sm.add_constant(X_train_df)
    X_test_ols = sm.add_constant(X_test_df)

    return X_train_ols, X_test_ols
X_train_ols, X_test_ols = add_constant(X_train_df, X_test_df)

In [41]:
def ols_model(y_train, X_train_ols):
    ols_model =sm.OLS(y_train, X_train_ols)
    fitted_model = ols_model.fit()
    summary = fitted_model.summary()
    return fitted_model, summary
fitted_model, summary = ols_model(y_train, X_train_ols)
display(summary)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         Standard_yield   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.624
Method:                 Least Squares   F-statistic:                     279.3
Date:                Sun, 16 Aug 2026   Prob (F-statistic):               0.00
Time:                        21:37:49   Log-Likelihood:                 5759.7
No. Observations:                4523   AIC:                        -1.146e+04
Df Residuals:                    4495   BIC:                        -1.128e+04
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      0.7905      0.255      3.097      0.002       0.290       1.291
Location_Rural_Akatsi     -0.0185      0.006     -3.282      0.001      -0.030      -0.007
Location_Rural_Amanzi     -0.0094      0.006     -1.585      0.113      -0.021       0.002
Location_Rural_Hawassa    -0.0184      0.005     -3.753      0.000      -0.028      -0.009
Location_Rural_Sokoto      0.0380      0.006      6.618      0.000       0.027       0.049
Soil_type_Loamy            0.1101      0.004     26.340      0.000       0.102       0.118
Soil_type_Peaty           -0.0689      0.008     -8.884      0.000      -0.084      -0.054
Soil_type_Rocky           -0.0469      0.005     -8.720      0.000      -0.057      -0.036
Soil_type_Silt            -0.0610      0.006     -9.461      0.000      -0.074      -0.048
Soil_type_Volcanic         0.0235      0.005      4.751      0.000       0.014       0.033
Crop_type_banana          -0.0712      0.004    -16.716      0.000      -0.080      -0.063
Crop_type_cassava         -0.0200      0.004     -4.965      0.000      -0.028      -0.012
Crop_type_coffee          -0.0734      0.004    -17.814      0.000      -0.081      -0.065
Crop_type_maize            0.0062      0.004      1.381      0.167      -0.003       0.015
Crop_type_potato           0.0724      0.004     19.820      0.000       0.065       0.080
Crop_type_rice             0.0803      0.006     13.945      0.000       0.069       0.092
Crop_type_tea              0.1010      0.005     21.789      0.000       0.092       0.110
Elevation                 -0.0001      0.000     -0.757      0.449      -0.000       0.000
Latitude                  -0.0038      0.001     -4.054      0.000      -0.006      -0.002
Longitude                 -0.0024      0.001     -2.612      0.009      -0.004      -0.001
Slope                      0.0015      0.001      1.350      0.177      -0.001       0.004
Rainfall                5.575e-05   3.11e-05      1.792      0.073   -5.25e-06       0.000
Min_temperature_C          0.0204      0.013      1.530      0.126      -0.006       0.046
Max_temperature_C         -0.0040      0.003     -1.488      0.137      -0.009       0.001
Ave_temps                  0.0082      0.005      1.530      0.126      -0.002       0.019
Soil_fertility            -0.4867      0.356     -1.367      0.172      -1.185       0.211
pH                         0.0245      0.002     12.613      0.000       0.021       0.028
Pollution_level           -0.2386      0.006    -38.863      0.000      -0.251      -0.227
Plot_size               6.487e-05      0.000      0.173      0.862      -0.001       0.001
==============================================================================
Omnibus:                       61.837   Durbin-Watson:                   2.004
Prob(Omnibus)